In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/data.csv")

df.head()

,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,ChannelId,Amount,Value,TransactionStartTime,PricingStrategy,FraudResult
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,ChannelId_3,1000.0,1000,2018-11-15T02:18:49Z,2,0
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-20.0,20,2018-11-15T02:19:08Z,2,0
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,ChannelId_3,500.0,500,2018-11-15T02:44:21Z,2,0
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,ChannelId_3,20000.0,21800,2018-11-15T03:32:55Z,2,0
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-644.0,644,2018-11-15T03:34:21Z,2,0


In [2]:
df["TransactionStartTime"] = pd.to_datetime(
    df["TransactionStartTime"]
)

In [3]:
snapshot_date = (
    df["TransactionStartTime"].max()
    + pd.Timedelta(days=1)
)

In [4]:
rfm = df.groupby("CustomerId").agg({

    "TransactionStartTime":
        lambda x:
        (snapshot_date - x.max()).days,

    "TransactionId":
        "count",

    "Amount":
        "sum"

}).reset_index()

In [5]:
rfm.columns = [
    "CustomerId",
    "Recency",
    "Frequency",
    "Monetary"
]

rfm.head()

,CustomerId,Recency,Frequency,Monetary
0,CustomerId_1,84,1,-10000.0
1,CustomerId_10,84,1,-10000.0
2,CustomerId_1001,90,5,20000.0
3,CustomerId_1002,26,11,4225.0
4,CustomerId_1003,12,6,20000.0


In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(
    rfm[
        [
            "Recency",
            "Frequency",
            "Monetary"
        ]
    ]
)

In [7]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=3,
    random_state=42
)

rfm["Cluster"] = kmeans.fit_predict(
    rfm_scaled
)

In [8]:
rfm.head()

,CustomerId,Recency,Frequency,Monetary,Cluster
0,CustomerId_1,84,1,-10000.0,0
1,CustomerId_10,84,1,-10000.0,0
2,CustomerId_1001,90,5,20000.0,0
3,CustomerId_1002,26,11,4225.0,2
4,CustomerId_1003,12,6,20000.0,2


In [9]:
cluster_summary = rfm.groupby(
    "Cluster"
)[
    [
        "Recency",
        "Frequency",
        "Monetary"
    ]
].mean()

cluster_summary

,Recency,Frequency,Monetary
Cluster,,,
0,61.859846,7.726699,8.172379e+04
1,29.000000,4091.000000,-1.049000e+08
2,12.716076,34.807692,2.726546e+05


Cluster 0: 
Highest Recency
Lowest Frequency
Low Monetary 

These customers are inactive and least engaged


HIGH RISK

In [15]:
high_risk_cluster = 1

In [11]:
rfm["is_high_risk"] = (
    rfm["Cluster"] == high_risk_cluster
).astype(int)

In [12]:
rfm["is_high_risk"].value_counts()

is_high_risk
0    2315
1    1427
Name: count, dtype: int64

In [13]:
df = df.merge(

    rfm[
        [
            "CustomerId",
            "is_high_risk"
        ]
    ],

    on="CustomerId",
    how="left"
)

In [14]:
df.head()

,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,ChannelId,Amount,Value,TransactionStartTime,PricingStrategy,FraudResult,is_high_risk
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,ChannelId_3,1000.0,1000,2018-11-15 02:18:49+00:00,2,0,0
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-20.0,20,2018-11-15 02:19:08+00:00,2,0,0
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,ChannelId_3,500.0,500,2018-11-15 02:44:21+00:00,2,0,1
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,ChannelId_3,20000.0,21800,2018-11-15 03:32:55+00:00,2,0,0
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-644.0,644,2018-11-15 03:34:21+00:00,2,0,0


K-Means clustering segmented customers into three behavioral groups based on Recency, Frequency, and Monetary value. Cluster 0 exhibited the highest recency, the lowest transaction frequency, and relatively low spending levels, indicating low customer engagement. Since disengaged customers are more likely to default or discontinue platform usage, Cluster 0 was designated as the high-risk segment. Customers in this cluster were assigned is_high_risk = 1, while customers in the remaining clusters were assigned 0